In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers

#Configuration(Sync with Verilog Testbench)
BIT_WIDTH = 9                  # 9-bit signed (-255 to +255) as in Lecture 10
MAX_INT = (1 << (BIT_WIDTH - 1)) - 1  # 255 for 9-bit; 127 for 8-bit
MEM_FORMAT = "bin"             # "bin" for $readmemb, "hex" for $readmemh
EPOCHS = 10
BATCH_SIZE = 64
SAMPLES_PER_CLASS = 10         # 10 samples x 10 digits = 100 balanced test samples
OUT_DIR = "mem_files"

os.makedirs(OUT_DIR, exist_ok=True)


#Loading and preprocessing MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize to [0, 1] and add channel dimension -> (N, 28, 28, 1)
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)


#Building and training the CNN
inputs = layers.Input(shape=(28, 28, 1))
x1 = layers.Conv2D(5, kernel_size=(3, 3), activation="relu", name="conv1")(inputs)
x2 = layers.MaxPooling2D(pool_size=(2, 2), name="pool1")(x1)
x3 = layers.Conv2D(10, kernel_size=(3, 3), activation="relu", name="conv2")(x2)
x4 = layers.MaxPooling2D(pool_size=(2, 2), name="pool2")(x3)
x5 = layers.Flatten(name="flat")(x4)
x6 = layers.Dropout(0.45, name="drop")(x5)
x7 = layers.Dense(128, activation="relu", name="dense1")(x6)
outputs = layers.Dense(10, activation="softmax", name="dense2")(x7)

model = models.Model(inputs=inputs, outputs=outputs)
model.summary()

model.compile(
    loss=keras.losses.categorical_crossentropy,
    optimizer=optimizers.Adam(learning_rate=0.001),
    metrics=["accuracy"],
)

history = model.fit(
    x_train,
    y_train_cat,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(x_test, y_test_cat),
)

# Plot Train vs Test Accuracy (Required for Part C(i))
plt.figure(figsize=(7, 5))
plt.plot(history.history["accuracy"], "ro-", label="Train")
plt.plot(history.history["val_accuracy"], "bo-", label="Test")
plt.title("CNN Accuracy on MNIST")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.savefig("mnist_cnn_accuracy.png", dpi=200)
plt.close()
print(f"Final Train Acc: {history.history['accuracy'][-1]*100:.2f}%")
print(f"Final Test Acc:  {history.history['val_accuracy'][-1]*100:.2f}%")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 26, 26, 5)      │            50 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 13, 13, 5)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 11, 11, 10)     │           460 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 5, 5, 10)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flat (Flatten)                  │ (None, 250)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop (Dropout)                  │ (None, 250)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 128)            │        32,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense2 (Dense)                  │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,928 (132.53 KB)

 Trainable params: 33,928 (132.53 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.8813 - loss: 0.3941 - val_accuracy: 0.9688 - val_loss: 0.1016
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9459 - loss: 0.1761 - val_accuracy: 0.9773 - val_loss: 0.0730
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9550 - loss: 0.1431 - val_accuracy: 0.9781 - val_loss: 0.0608
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9607 - loss: 0.1242 - val_accuracy: 0.9819 - val_loss: 0.0513
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9631 - loss: 0.1118 - val_accuracy: 0.9845 - val_loss: 0.0467
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9672 - loss: 0.1020 - val_accuracy: 0.9850 - val_loss: 0.0404
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9695 - loss: 0.0950 - val_accuracy: 0.9845 - val_loss: 0.0398
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9723 - loss: 0.0886 - val_accuracy: 

In [ ]:
#Quantization and .mem export helpers
def quantize_layer(w, b, max_int=MAX_INT):
    """Scale weights and biases jointly to [-max_int, +max_int] integers (Lec 10 method)."""
    max_val = max(np.max(np.abs(w)), np.max(np.abs(b)))
    if max_val == 0:
        return np.zeros_like(w, dtype=int), np.zeros_like(b, dtype=int), 1.0
    scale = max_int / max_val
    w_q = np.round(w * scale).astype(int)
    b_q = np.round(b * scale).astype(int)
    return w_q, b_q, scale


def write_mem_file(filepath, array_1d, bit_width=BIT_WIDTH, fmt=MEM_FORMAT):
    """Write 1D integer array to .mem file in 2's complement binary or hex."""
    mask = (1 << bit_width) - 1
    hex_digits = (bit_width + 3) // 4
    with open(filepath, "w") as f:
        for val in array_1d:
            twos_comp = int(val) & mask
            if fmt == "bin":
                f.write(format(twos_comp, f"0{bit_width}b") + "\n")
            else:
                f.write(format(twos_comp, f"0{hex_digits}x") + "\n")

In [ ]:
#Extract and format weights for systolic array
# Keras Conv2D kernel shape: (kH, kW, C_in, C_out)
# For systolic array B matrix: (kH * kW * C_in) x C_out
# Each column is one flattened filter.

for layer_name in ["conv1", "conv2", "dense1", "dense2"]:
    w, b = model.get_layer(layer_name).get_weights()
    w_q, b_q, _ = quantize_layer(w, b)

    if "conv" in layer_name:
        # Transpose to (C_in, kH, kW, C_out) then reshape to (C_in*kH*kW, C_out)
        # so each filter is flattened channel-by-channel, row by row
        c_out = w_q.shape[-1]
        w_mat = np.transpose(w_q, (2, 0, 1, 3)).reshape(-1, c_out)
    else:
        # Dense weight shape is already (in_features, out_features)
        w_mat = w_q

    # Flatten row-major (default) into .mem.
    #Need to change to w_mat.flatten(order='F') if Verilog reads column major
    write_mem_file(os.path.join(OUT_DIR, f"{layer_name}_weights.mem"), w_mat.flatten(order="C"))
    write_mem_file(os.path.join(OUT_DIR, f"{layer_name}_biases.mem"), b_q.flatten())
    print(f"Exported {layer_name}: weight matrix {w_mat.shape}, biases {b_q.shape}")

Exported conv1: weight matrix (9, 5), biases (5,)
Exported conv2: weight matrix (45, 10), biases (10,)
Exported dense1: weight matrix (250, 128), biases (128,)
Exported dense2: weight matrix (128, 10), biases (10,)


In [ ]:
#Export balanced 100 images and test labels
selected_imgs = []
selected_labels = []

for digit in range(10):
    idxs = np.where(y_test == digit)[0][:SAMPLES_PER_CLASS]
    selected_imgs.append(x_test[idxs])
    selected_labels.append(y_test[idxs])

test_imgs_100 = np.concatenate(selected_imgs, axis=0)      # (100, 28, 28, 1)
test_labels_100 = np.concatenate(selected_labels, axis=0)  # (100,)

#Scaling the pixel values from 0 to 1 to 0 to MAX_INT
test_imgs_q = np.round(test_imgs_100 * MAX_INT).astype(int)

write_mem_file(os.path.join(OUT_DIR, "test_images_100.mem"), test_imgs_q.flatten())
with open(os.path.join(OUT_DIR, "test_labels_100.mem"), "w") as f:
    for lbl in test_labels_100:
        f.write(f"{int(lbl)}\n")

print(f"Exported 100 balanced test images and labels to '{OUT_DIR}/'.")

Exported 100 balanced test images and labels to 'mem_files/'.
